In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

from spagrn.regulatory_network import InferNetwork as irn

In [ ]:
adata = sc.read_h5ad("fetal_brain_hongtaoyu/0318_fetralbrain_rawcount_bin100/9_D03466F1G2_WT202403310042.h5ad")
adata_obs = pd.read_csv("fetal_brain_hongtaoyu/bin100_label/9_D03466F1G2_WT202403310042.csv",index_col=[0])
adata_obs 

In [ ]:
adata = adata[adata_obs.index,:]
adata.obs = adata_obs
adata

In [ ]:
sc.pl.spatial(adata,color=['region','region_h2'],spot_size=100)

In [ ]:
adata = irn.preprocess(adata, min_genes=10, min_cells=int(len(adata.obs_names)*0.01), min_counts=10, max_gene_num=20000)
X = adata.X
if sp.isspmatrix_csc(X):
    adata.layers['count'] = X.copy()
else:
    if sp.issparse(X):
        adata.layers['count'] = X.tocsc().copy()
    else:
        # dense numpy array -> convert to CSC sparse matrix
        adata.layers['count'] = sp.csc_matrix(X)
del X
print(adata)

In [ ]:
adata.X.max()

In [ ]:
tfs_fn = 'GRN_resource/hs_hgnc_tfs.txt'
database_fn = 'GRN_resource/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
motif_anno_fn = 'GRN_resource/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl'

niche_human = pd.read_csv('GRN_resource/lr_network_human.csv')
niche_mouse = pd.read_csv('GRN_resource/lr_network_mouse.csv')
niches = pd.concat([niche_mouse, niche_human])
print(niches)

## GPU

In [ ]:
grn = irn(adata, project_name='human_organoid')
grn.add_params({'prune_auc_threshold': 0.05, 'rank_threshold': 9000, 'auc_threshold': 0.05})

grn.infer(database_fn,
          motif_anno_fn,
          tfs_fn,
          niche_df=niches,
          gene_list=None,
          num_workers=15,
          cache=True,
          output_dir='Output/GRN_output/test_large_gpu',
          save_tmp=True,
          layers='count',
          latent_obsm_key='spatial',
          model='danb',
          n_neighbors=10,
          methods=['FDR_I','FDR_C','FDR_G'],
          operation='intersection',
          mode='geary',
          torch_device='cuda:6',
          pair_batch_size=1024,
          bivariate_pair_batch_size=512,
          cluster_label='region')

## CPU

In [ ]:
grn = irn(adata, project_name='human_organoid')
grn.add_params({'prune_auc_threshold': 0.05, 'rank_threshold': 9000, 'auc_threshold': 0.05})

grn.infer(database_fn,
          motif_anno_fn,
          tfs_fn,
          niche_df=niches,
          gene_list=None,
          num_workers=15,
          cache=True,
          output_dir='Output/GRN_output/test_large',
          save_tmp=True,
          layers='count',
          latent_obsm_key='spatial',
          model='danb',
          n_neighbors=10,
          methods=['FDR_I','FDR_C','FDR_G'],
          operation='intersection',
          mode='geary',
          cluster_label='region')